# Merchant fraud risk

This notebook uses `merchant_fraud_probability` to decide which merchants look dodgy. The table has only 114 scored merchant-days across 61 merchants, so no model is fitted on the score values directly. Instead the notebook:

1. Checks what drives a merchant-day score.
2. Builds merchant-level summary features from all transactions in the scoring period. These describe transaction spread, daily volatility, spikes and consumer-flag exposure.
3. Fits a classifier for "does this merchant get fraud-scored at all?", with cross-validation.
4. Combines known scores and model suspicion into a merchant risk table, saved to `processed/merchant_risk`.

As with consumers, a score marks a pre-screened day (the lowest score is 18%). Having no score is not proof a merchant is clean.

## 1. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, Window, functions as F
from scipy.stats import spearmanr
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import average_precision_score, roc_auc_score
from xgboost import XGBClassifier

# Works when launched from the repository, project2, or notebooks directory.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if p.name == "project2"), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd() / "project2"
PROCESSED_PATH = PROJECT_ROOT / "data" / "tables" / "processed"
if not PROCESSED_PATH.is_dir():
    raise FileNotFoundError("Set PROJECT_ROOT to your project2 directory.")
RANDOM_STATE = 42
CUTOFFS = [30, 50, 70]
# Scores exist only in this window; features are computed over the same period.
START, END = "2021-02-28", "2022-02-27"
# Without a cap, XGBoost oversubscribes the CPU alongside Spark and slows to a crawl.
XGB_THREADS = 4
spark = SparkSession.builder.config("spark.driver.memory", "6g").getOrCreate()

## 2. Merchant-day aggregates

In [ ]:
trans = (
    spark.read.parquet(str(PROCESSED_PATH / "transactions_clean"))
    .withColumn("order_date", F.to_date("order_datetime"))
    .where(F.col("order_date").between(START, END))
    .withColumn("day_txn_count", F.count("*").over(Window.partitionBy("user_id", "order_date")))
)

merchant_w = Window.partitionBy("merchant_abn")
merchant_day = (
    trans.groupBy("merchant_abn", "order_date")
    .agg(F.sum("dollar_value").alias("revenue"), F.count("*").alias("txn_count"),
         F.mean("dollar_value").alias("avg_value"), F.max("dollar_value").alias("max_value"))
    # Each day relative to that merchant's typical (median) day.
    .withColumn("revenue_ratio", F.col("revenue") / F.percentile_approx("revenue", 0.5).over(merchant_w))
    .withColumn("count_ratio", F.col("txn_count") / F.percentile_approx("txn_count", 0.5).over(merchant_w))
    .withColumn("avg_value_ratio", F.col("avg_value") / F.percentile_approx("avg_value", 0.5).over(merchant_w))
    .withColumn("typical_revenue", F.percentile_approx("revenue", 0.5).over(merchant_w))
).cache()

## 3. What drives a merchant-day score?

This correlates each scored day's features with its score.

In [ ]:
merchant_scores = (spark.read.parquet(str(PROCESSED_PATH / "merchant_fraud_probability"))
                   .withColumn("order_date", F.to_date("order_datetime")).drop("order_datetime"))
scored_days = merchant_scores.join(merchant_day, ["merchant_abn", "order_date"], "left").toPandas()
assert scored_days["revenue"].notna().all(), "every scored day should match transactions"

DAY_COLS = ["revenue", "txn_count", "avg_value", "revenue_ratio", "count_ratio",
            "avg_value_ratio", "typical_revenue"]
pd.Series({c: spearmanr(scored_days[c], scored_days["fraud_probability"]).statistic
           for c in DAY_COLS}, name="Spearman with score").round(2)

- **Relative spikes drive the score.** The score tracks revenue and average value relative to the merchant's usual day. Raw revenue alone does not.
- **The highest scores are single oversized transactions.** They fall on days with a normal transaction count but a much larger average value.
- **Scored days are also large in absolute terms.** Most have $40K–$60K of revenue, versus a typical merchant day under $1K.
- **Not every spike gets scored.** The cell below shows that scored days are only a subset of similar days.

In [ ]:
day_pd = merchant_day.select("revenue", "revenue_ratio").toPandas()
print(f"Merchant-days with revenue >= $40K: {(day_pd['revenue'] >= 40_000).sum():,}; "
      f"scored: {(scored_days['revenue'] >= 40_000).sum():,}")

## 4. Merchant-level summary features

The features are computed over the scoring window. None of them use the merchant scores.

| Group | Features |
|---|---|
| Transaction spread | coefficient of variation (standard deviation ÷ mean), 99th percentile ÷ median, max ÷ median |
| Daily volatility | coefficient of variation of daily revenue and of daily transaction count |
| Spikes | largest daily revenue ratio and largest average-value ratio; number of days where either is ≥ 3× the merchant's median |
| Size | transaction count, mean transaction value, median and max daily revenue, active days, take rate |
| Customers | repeat-customer share |
| Consumer-flag exposure | transactions on consumer-scored days with score ≥ 30 and ≥ 50, weighted 1 ÷ the user's transactions that day, as in `fraud_classifier.ipynb` |

In [ ]:
consumer_score = F.col("consumer_fraud_probability")
day_weight = 1.0 / F.col("day_txn_count")
txn_stats = trans.groupBy("merchant_abn").agg(
    F.count("*").alias("txn_count"), F.countDistinct("user_id").alias("distinct_users"),
    F.countDistinct("order_date").alias("active_days"),
    F.mean("dollar_value").alias("txn_mean"), F.stddev("dollar_value").alias("txn_std"),
    F.percentile_approx("dollar_value", 0.5).alias("txn_median"),
    F.percentile_approx("dollar_value", 0.99).alias("txn_p99"), F.max("dollar_value").alias("txn_max"),
    *[F.sum(F.when(consumer_score >= k, day_weight).otherwise(0)).alias(f"consumer_flag_ge{k}")
      for k in (30, 50)],
)
day_stats = merchant_day.groupBy("merchant_abn").agg(
    F.mean("revenue").alias("day_rev_mean"), F.stddev("revenue").alias("day_rev_std"),
    F.percentile_approx("revenue", 0.5).alias("day_rev_median"), F.max("revenue").alias("day_rev_max"),
    F.mean("txn_count").alias("day_n_mean"), F.stddev("txn_count").alias("day_n_std"),
    F.max("revenue_ratio").alias("day_rev_ratio_max"),
    F.max("avg_value_ratio").alias("day_avg_value_ratio_max"),
    F.sum((F.col("revenue_ratio") >= 3).cast("int")).alias("days_rev_ratio_ge3"),
    F.sum((F.col("avg_value_ratio") >= 3).cast("int")).alias("days_avg_value_ratio_ge3"),
)
merchant = txn_stats.join(day_stats, "merchant_abn").toPandas()

labels = scored_days.groupby("merchant_abn")["fraud_probability"].agg(
    merchant_score_max="max", merchant_scored_days="size").reset_index()
merchant_info = pd.read_parquet(PROCESSED_PATH / "merchants_clean",
                                columns=["merchant_abn", "name", "revenue_code", "take_rate"])
merchant = (merchant.merge(labels, on="merchant_abn", how="left")
            .merge(merchant_info, on="merchant_abn", how="left"))
merchant["flagged"] = merchant["merchant_score_max"].notna().astype(int)

merchant["txn_cv"] = merchant["txn_std"] / merchant["txn_mean"]
merchant["txn_p99_to_median"] = merchant["txn_p99"] / merchant["txn_median"]
merchant["txn_max_to_median"] = merchant["txn_max"] / merchant["txn_median"]
merchant["day_rev_cv"] = merchant["day_rev_std"] / merchant["day_rev_mean"]
merchant["day_n_cv"] = merchant["day_n_std"] / merchant["day_n_mean"]
merchant["repeat_share"] = 1 - merchant["distinct_users"] / merchant["txn_count"]
merchant["consumer_flag_share_ge50"] = merchant["consumer_flag_ge50"] / merchant["txn_count"]
for col in ["txn_count", "txn_mean", "day_rev_median", "day_rev_max",
            "consumer_flag_ge30", "consumer_flag_ge50"]:
    merchant[f"log_{col}"] = np.log1p(merchant[col])

print(f"{len(merchant):,} merchants active in the window; {merchant['flagged'].sum()} have a merchant score "
      f"({merchant.loc[merchant['flagged'] == 1, 'revenue_code'].isna().sum()} of those are missing from merchants_clean)")

In [ ]:
SPREAD = ["txn_cv", "txn_p99_to_median", "txn_max_to_median", "day_rev_cv", "day_n_cv",
          "day_rev_ratio_max", "day_avg_value_ratio_max", "days_rev_ratio_ge3", "days_avg_value_ratio_ge3"]
SIZE = ["log_txn_count", "log_txn_mean", "log_day_rev_median", "log_day_rev_max",
        "active_days", "repeat_share", "take_rate"]
CONSUMER = ["log_consumer_flag_ge30", "log_consumer_flag_ge50", "consumer_flag_share_ge50"]

merchant.groupby("flagged")[SPREAD + ["txn_mean", "day_rev_median", "day_rev_max", "consumer_flag_ge50"]] \
    .median().T.rename(columns={0: "unscored (median)", 1: "scored (median)"}).round(2)

## 5. Which merchants get scored?

The target is whether a merchant has at least one merchant score. There are about 61 positives out of about 4,400 merchants. Each feature set is evaluated with 5-fold stratified cross-validation, repeated over 5 seeds. The PR-AUC of a random ranking equals the prevalence.

In [ ]:
def make_model():
    return XGBClassifier(n_estimators=300, learning_rate=0.03, max_depth=3, min_child_weight=2,
                         subsample=0.8, colsample_bytree=0.8, n_jobs=XGB_THREADS,
                         random_state=RANDOM_STATE)

def out_of_fold(features, seeds=range(5)):
    # Out-of-fold predictions, averaged over repeats, so no merchant is scored
    # by a model that saw its own label.
    y = merchant["flagged"]
    preds = [cross_val_predict(make_model(), merchant[features], y, method="predict_proba",
                               cv=StratifiedKFold(5, shuffle=True, random_state=s))[:, 1] for s in seeds]
    ap = [average_precision_score(y, p) for p in preds]
    return np.mean(preds, axis=0), {"PR-AUC": np.mean(ap), "PR-AUC sd": np.std(ap),
                                    "ROC-AUC": np.mean([roc_auc_score(y, p) for p in preds])}

results, oof = {}, {}
for name, features in {"spread only": SPREAD, "spread + size": SPREAD + SIZE,
                       "spread + size + consumer flags": SPREAD + SIZE + CONSUMER}.items():
    oof[name], results[name] = out_of_fold(features)
results = pd.DataFrame(results).T.assign(prevalence=merchant["flagged"].mean())
results.round(3)

- **Spread features alone** rank flagged merchants about 10× better than random, but they're weak on their own.
- **Size is what separates scored merchants.** They are big-ticket merchants: much higher mean transaction and typical daily revenue. The scoring appears to fire on large-revenue spike days, and small merchants rarely reach those amounts.
- **Consumer-flag exposure** adds a little on top.

**Caution for the ranking.** A model that relies on size will also mark large, legitimate, high-value merchants as risky. Use the probability as a prompt for review, and keep size out of any penalty applied to the ranking itself.

In [ ]:
FEATURES = SPREAD + SIZE + CONSUMER
merchant["p_flagged"] = oof["spread + size + consumer flags"]
final_model = make_model().fit(merchant[FEATURES], merchant["flagged"])
gain = pd.Series(final_model.get_booster().get_score(importance_type="gain")).sort_values()
gain.tail(12).plot.barh(figsize=(7, 4), title="XGBoost gain, P(merchant is scored)")
plt.tight_layout()
plt.show()

## 6. Merchant risk table

| `risk_tier` | Rule |
|---|---|
| `high` | highest merchant score ≥ 70 |
| `flagged` | highest merchant score 50–70 |
| `watch` | highest merchant score 30–50 |
| `scored_low` | scored, but every score < 30 |
| `suspect` | never scored, but out-of-fold P(scored) ≥ `SUSPECT_THRESHOLD` |
| `none` | everything else |

The suspect threshold is a default. Check how many merchants it picks up and adjust it.

In [ ]:
SUSPECT_THRESHOLD = 0.5
s = merchant["merchant_score_max"]
merchant["risk_tier"] = np.select(
    [s >= 70, s >= 50, s >= 30, s.notna(), merchant["p_flagged"] >= SUSPECT_THRESHOLD],
    ["high", "flagged", "watch", "scored_low", "suspect"], "none")
print(merchant["risk_tier"].value_counts().reindex(
    ["high", "flagged", "watch", "scored_low", "suspect", "none"]).to_string())

RISK_COLS = ["merchant_abn", "name", "revenue_code", "risk_tier", "merchant_score_max",
             "merchant_scored_days", "p_flagged", "consumer_flag_ge30", "consumer_flag_ge50",
             "consumer_flag_share_ge50", *SPREAD, "txn_count", "txn_mean", "day_rev_median"]
merchant_risk = merchant[RISK_COLS].sort_values(["merchant_score_max", "p_flagged"], ascending=False)
merchant_risk.head(15)

In [ ]:
merchant_risk[merchant_risk["risk_tier"] == "suspect"].sort_values("p_flagged", ascending=False).head(15)

In [ ]:
spark.createDataFrame(merchant_risk).write.mode("overwrite").parquet(str(PROCESSED_PATH / "merchant_risk"))

## Interpretation and limitations

- **Labels are sparse.** 61 merchants have a score, and only a handful score ≥ 70. Treat the tiers as flags for review, not proof of fraud.
- **Unscored ≠ clean.** Many high-revenue spike days were never scored. The `suspect` tier is how this notebook surfaces merchants that look like scored ones.
- **Size confound.** The classifier leans on merchant size. Keep `p_flagged` separate from any size-related term in the ranking, or large merchants will be double-penalised.
- **Unmatched merchants.** Some scored merchants are missing from `merchants_clean`, so they have no name or revenue code and drop out of any ranking that joins on it.
- **Look-ahead features.** Features cover the whole scoring window, including days after a given score. That's fine for describing merchants, but a live system would need them computed from past data only.